In [0]:

bronze_path = "/Volumes/dbw_agricoop_poc/ingestion/events_bronze/"

df_bronze = spark.read.json(
    bronze_path + "partition=2/*.json"
)

display(df_bronze)

print("Nombre d'événements :", df_bronze.count())
df_bronze.printSchema()


In [0]:
from pyspark.sql.functions import col, to_timestamp, to_json

df_silver = (
    df_bronze
    .filter(col("pseudonymise") == True)
    .filter(col("event_id").isNotNull())
    .dropDuplicates(["event_id"])
    .select(
        "event_id",
        "coop_id",
        "type",
        "version",
        to_timestamp("occurred_at").alias("event_timestamp"),
        to_json(col("payload")).alias("payload_json")
    )
)

display(df_silver)

print("Événements Silver :", df_silver.count())

In [0]:
silver_path = "abfss://silver@stagricoopdata26.dfs.core.windows.net/events_delta/"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("path", silver_path)
    .saveAsTable("dbw_agricoop_poc.silver.events")
)

print("Table Silver créée avec succès")

In [0]:
df = spark.table("dbw_agricoop_poc.silver.events")

display(df)
print("Nombre de lignes :", df.count())

In [0]:
spark.sql("""
DESCRIBE DETAIL dbw_agricoop_poc.silver.events
""").select("format", "location", "numFiles").show(truncate=False)

In [0]:
from pyspark.sql.functions import count

df_silver.groupBy("coop_id", "type").agg(
    count("*").alias("nb_evenements")
).show(truncate=False)

In [0]:

from pyspark.sql import functions as F

# Lire la table Silver déjà enregistrée
silver = spark.table("dbw_agricoop_poc.silver.events")

# Structure des lignes de commande
schema_lignes = """
array<struct<
    code_produit:string,
    quantite:int,
    prix_unitaire_mad:double
>>
"""

# Construire les données Gold
df_gold_ventes = (
    silver
    .filter(F.col("type") == "commande_creee")
    .withColumn(
        "lignes",
        F.from_json(
            F.get_json_object("payload_json", "$.lignes"),
            schema_lignes
        )
    )
    .withColumn("ligne", F.explode("lignes"))
    .select(
        "event_id",
        "coop_id",
        F.to_date("event_timestamp").alias("jour"),
        F.col("ligne.code_produit").alias("produit"),
        F.col("ligne.quantite").alias("quantite"),
        (
            F.col("ligne.quantite") *
            F.col("ligne.prix_unitaire_mad")
        ).alias("ca_mad")
    )
)

display(df_gold_ventes)


In [0]:

gold_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/ventes_delta/"

(
    df_gold_ventes.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_path)
    .saveAsTable("dbw_agricoop_poc.gold.ventes")
)

print("Table Gold ventes créée avec succès.")


In [0]:

df = spark.table("dbw_agricoop_poc.gold.ventes")

display(df)

print("Nombre de ventes :", df.count())

spark.sql("""
DESCRIBE DETAIL dbw_agricoop_poc.gold.ventes
""").select("format", "location").show(truncate=False)


In [0]:

from pyspark.sql import functions as F

silver = spark.table("dbw_agricoop_poc.silver.events")

df_gold_stock = (
    silver
    .filter(F.col("type") == "stock_modifie")
    .select(
        "event_id",
        "coop_id",
        F.to_date("event_timestamp").alias("jour"),
        F.get_json_object(
            "payload_json", "$.code_produit"
        ).alias("produit"),
        F.get_json_object(
            "payload_json", "$.type_mouvement"
        ).alias("type_mouvement"),
        F.get_json_object(
            "payload_json", "$.quantite"
        ).cast("int").alias("quantite")
    )
)

display(df_gold_stock)
print("Mouvements de stock :", df_gold_stock.count())


In [0]:

gold_stock_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/mouvements_stock_delta/"

(
    df_gold_stock.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_stock_path)
    .saveAsTable("dbw_agricoop_poc.gold.mouvements_stock")
)

print("Table Gold mouvements_stock créée avec succès.")


In [0]:

df = spark.table("dbw_agricoop_poc.gold.mouvements_stock")

display(df)
print("Nombre de mouvements :", df.count())


In [0]:

from pyspark.sql import functions as F

ventes = spark.table("dbw_agricoop_poc.gold.ventes")
stocks = spark.table("dbw_agricoop_poc.gold.mouvements_stock")

# Indicateurs commerciaux par coopérative
kpi_ventes = ventes.groupBy("coop_id").agg(
    F.sum("ca_mad").alias("ca_total_mad"),
    F.sum("quantite").alias("quantites_vendues"),
    F.countDistinct("event_id").alias("nb_commandes")
)

# Indicateurs opérationnels par coopérative
kpi_stocks = stocks.groupBy("coop_id").agg(
    F.countDistinct("event_id").alias("nb_mouvements_stock")
)

# Réunir les KPI sans multiplier les lignes
df_kpi = kpi_ventes.join(
    kpi_stocks, on="coop_id", how="full_outer"
).fillna({
    "ca_total_mad": 0,
    "quantites_vendues": 0,
    "nb_commandes": 0,
    "nb_mouvements_stock": 0
})

display(df_kpi)


In [0]:

(
    df_kpi.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        "abfss://gold@stagricoopdata26.dfs.core.windows.net/kpi_cooperative_delta/"
    )
    .saveAsTable("dbw_agricoop_poc.gold.kpi_cooperative")
)

print("Table KPI Gold créée avec succès.")


In [0]:
gold_kpi_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/kpi_cooperative_delta/"

(
    df_kpi.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_kpi_path)
    .saveAsTable("dbw_agricoop_poc.gold.kpi_cooperative")
)

print("Table KPI Gold créée avec succès.")

In [0]:
display(
    spark.table("dbw_agricoop_poc.gold.kpi_cooperative")
)

In [0]:
gold_kpi_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/kpi_cooperative_delta/"

(
    df_kpi.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_kpi_path)
    .saveAsTable("dbw_agricoop_poc.gold.kpi_cooperative")
)

print("Table KPI Gold créée avec succès.")

In [0]:
display(
    spark.table("dbw_agricoop_poc.gold.kpi_cooperative")
)

In [0]:
tables = ["ventes", "mouvements_stock", "kpi_cooperative"]

for name in tables:
    table = f"dbw_agricoop_poc.gold.{name}"

    print(f"\nTable : {table}")
    print("Nombre de lignes :", spark.table(table).count())

    spark.sql(f"DESCRIBE DETAIL {table}") \
        .select("format", "location") \
        .show(truncate=False)

In [0]:

bronze_path = "/Volumes/dbw_agricoop_poc/ingestion/events_bronze/"

# Lire les fichiers JSON de toutes les partitions
df_bronze = spark.read.json(
    bronze_path + "partition=*/*.json"
)

display(df_bronze)

print("Nombre total d'événements :", df_bronze.count())

df_bronze.groupBy("coop_id", "type").count().show()


In [0]:

from pyspark.sql import functions as F

df_bronze.agg(
    F.count("*").alias("total_evenements"),
    F.countDistinct("event_id").alias("evenements_uniques"),
    F.sum(
        F.when(F.col("event_id").isNull(), 1).otherwise(0)
    ).alias("ids_manquants")
).show()

# Afficher les event_id présents plusieurs fois
df_bronze.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)


In [0]:

from pyspark.sql import functions as F

df_silver = (
    df_bronze
    .filter(F.col("pseudonymise") == True)
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("coop_id").isNotNull())
    .dropDuplicates(["event_id"])
    .select(
        "event_id",
        "coop_id",
        "type",
        "version",
        F.to_timestamp("occurred_at").alias("event_timestamp"),
        F.to_json("payload").alias("payload_json")
    )
)

print("Événements Silver :", df_silver.count())
df_silver.groupBy("type").count().show()


In [0]:

silver_path = "abfss://silver@stagricoopdata26.dfs.core.windows.net/events_delta/"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", silver_path)
    .saveAsTable("dbw_agricoop_poc.silver.events")
)

print(
    "Événements Silver enregistrés :",
    spark.table("dbw_agricoop_poc.silver.events").count()
)


In [0]:

from pyspark.sql import functions as F

silver = spark.table("dbw_agricoop_poc.silver.events")

schema_lignes = """
array<struct<
    code_produit:string,
    quantite:int,
    prix_unitaire_mad:double
>>
"""

df_gold_ventes = (
    silver
    .filter(F.col("type") == "commande_creee")
    .withColumn(
        "lignes",
        F.from_json(
            F.get_json_object("payload_json", "$.lignes"),
            schema_lignes
        )
    )
    .withColumn("ligne", F.explode("lignes"))
    .select(
        "event_id",
        "coop_id",
        F.to_date("event_timestamp").alias("jour"),
        F.col("ligne.code_produit").alias("produit"),
        F.col("ligne.quantite").alias("quantite"),
        (
            F.col("ligne.quantite") *
            F.col("ligne.prix_unitaire_mad")
        ).alias("ca_mad")
    )
)

# Vérifier les résultats par coopérative
display(
    df_gold_ventes.groupBy("coop_id").agg(
        F.countDistinct("event_id").alias("nb_commandes"),
        F.sum("quantite").alias("quantites_vendues"),
        F.round(F.sum("ca_mad"), 2).alias("ca_total_mad")
    )
)


In [0]:

gold_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/ventes_delta/"

(
    df_gold_ventes.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_path)
    .saveAsTable("dbw_agricoop_poc.gold.ventes")
)

print("Table Gold ventes actualisée.")

print(
    "Nombre de commandes :",
    spark.table("dbw_agricoop_poc.gold.ventes")
         .select("event_id").distinct().count()
)


In [0]:

from pyspark.sql import functions as F

silver = spark.table("dbw_agricoop_poc.silver.events")

df_gold_stock = (
    silver
    .filter(F.col("type") == "stock_modifie")
    .select(
        "event_id",
        "coop_id",
        F.to_date("event_timestamp").alias("jour"),
        F.get_json_object(
            "payload_json", "$.code_produit"
        ).alias("produit"),
        F.get_json_object(
            "payload_json", "$.type_mouvement"
        ).alias("type_mouvement"),
        F.get_json_object(
            "payload_json", "$.quantite"
        ).cast("int").alias("quantite")
    )
)

display(df_gold_stock)

print("Nombre de mouvements :", df_gold_stock.count())

df_gold_stock.groupBy("type_mouvement").agg(
    F.count("*").alias("nb_mouvements"),
    F.sum("quantite").alias("quantite_totale")
).show()


In [0]:

gold_stock_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/mouvements_stock_delta/"

(
    df_gold_stock.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_stock_path)
    .saveAsTable("dbw_agricoop_poc.gold.mouvements_stock")
)

print("Table mouvements_stock actualisée.")

print(
    "Nombre de mouvements :",
    spark.table("dbw_agricoop_poc.gold.mouvements_stock").count()
)


In [0]:

from pyspark.sql import functions as F

# Lire les deux tables Gold actualisées
ventes = spark.table("dbw_agricoop_poc.gold.ventes")
stocks = spark.table("dbw_agricoop_poc.gold.mouvements_stock")

# KPI commerciaux
kpi_ventes = ventes.groupBy("coop_id").agg(
    F.sum("ca_mad").alias("ca_total_mad"),
    F.sum("quantite").alias("quantites_vendues"),
    F.countDistinct("event_id").alias("nb_commandes")
)

# KPI de stock
kpi_stocks = stocks.groupBy("coop_id").agg(
    F.countDistinct("event_id").alias("nb_mouvements_stock")
)

# Réunir les indicateurs par coopérative
df_kpi = (
    kpi_ventes
    .join(kpi_stocks, on="coop_id", how="full_outer")
    .fillna(0, subset=[
        "ca_total_mad",
        "quantites_vendues",
        "nb_commandes",
        "nb_mouvements_stock"
    ])
)

display(df_kpi)


In [0]:

(
    df_kpi.write
    .format("delta")
    .mode("overwrite")
    .option(
        "path",
        "abfss://gold@stagricoopdata26.dfs.core.windows.net/kpi_cooperative_delta/"
    )
    .saveAsTable("dbw_agricoop_poc.gold.kpi_cooperative")
)

print("KPI Gold actualisés avec succès.")
display(spark.table("dbw_agricoop_poc.gold.kpi_cooperative"))


In [0]:

from pyspark.sql import functions as F

silver = spark.table("dbw_agricoop_poc.silver.events")

df_gold_qr = (
    silver
    .filter(F.col("type") == "qr_scanne")
    .select(
        "event_id",
        "coop_id",
        F.col("event_timestamp").alias("date_scan"),
        F.get_json_object(
            "payload_json", "$.qr_id"
        ).alias("qr_id"),
        F.get_json_object(
            "payload_json", "$.scan_id"
        ).alias("scan_id"),
        F.get_json_object(
            "payload_json", "$.ville"
        ).alias("ville"),
        F.get_json_object(
            "payload_json", "$.niveau"
        ).alias("niveau")
    )
)

display(df_gold_qr)
print("Nombre de scans QR :", df_gold_qr.count())


In [0]:

gold_qr_path = "abfss://gold@stagricoopdata26.dfs.core.windows.net/scans_qr_delta/"

(
    df_gold_qr.write
    .format("delta")
    .mode("overwrite")
    .option("path", gold_qr_path)
    .saveAsTable("dbw_agricoop_poc.gold.scans_qr")
)

print("Table Gold scans_qr créée avec succès.")

display(
    spark.table("dbw_agricoop_poc.gold.scans_qr")
)


In [0]:

tables = [
    "ventes",
    "mouvements_stock",
    "kpi_cooperative",
    "scans_qr"
]

for nom in tables:
    table = f"dbw_agricoop_poc.gold.{nom}"

    df = spark.table(table)
    details = spark.sql(f"DESCRIBE DETAIL {table}")

    print(f"\nTable : {nom}")
    print("Nombre de lignes :", df.count())

    details.select(
        "format", "location"
    ).show(truncate=False)


In [0]:
historique = "/Volumes/dbw_agricoop_poc/ingestion/historique_bronze/"

df_ventes_hist = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(historique + "v_ventes_jour.csv")
)

print("Nombre de lignes historiques :", df_ventes_hist.count())
df_ventes_hist.printSchema()
display(df_ventes_hist.limit(5))

In [0]:

from pyspark.sql import functions as F

# 1. Volume et période couverte
df_ventes_hist.agg(
    F.count("*").alias("nb_lignes"),
    F.countDistinct("produit").alias("nb_produits"),
    F.countDistinct("coop_id").alias("nb_cooperatives"),
    F.min("jour").alias("premiere_vente"),
    F.max("jour").alias("derniere_vente"),
    F.round(F.sum("ca_mad"), 2).alias("ca_total_mad")
).show(truncate=False)

# 2. Chiffre d'affaires par produit
df_ventes_hist.groupBy("produit").agg(
    F.sum("quantite").alias("quantites_vendues"),
    F.round(F.sum("ca_mad"), 2).alias("ca_mad")
).orderBy(F.desc("ca_mad")).show(20, truncate=False)


In [0]:

table_name = "dbw_agricoop_poc.gold.historique_ventes_jour"

historique_gold_path = (
    "abfss://gold@stagricoopdata26.dfs.core.windows.net/"
    "historique_ventes_jour_delta/"
)

# Éviter de remplacer une table déjà existante
if not spark.catalog.tableExists(table_name):
    (
        df_ventes_hist.write
        .format("delta")
        .mode("overwrite")
        .option("path", historique_gold_path)
        .saveAsTable(table_name)
    )
    print("Table historique créée avec succès.")
else:
    print("La table existe déjà : aucune modification.")

# Vérification
hist = spark.table(table_name)

print("Nombre de lignes :", hist.count())
print("Nombre de produits :", hist.select("produit").distinct().count())


In [0]:

historique = "/Volumes/dbw_agricoop_poc/ingestion/historique_bronze/"

fichiers = {
    "kpi_mensuel": "v_kpi_mensuel.csv",
    "stock_produit": "v_stock_produit.csv",
    "prevision_vs_ventes": "v_prevision_vs_ventes.csv"
}

dataframes = {}

for nom, fichier in fichiers.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(historique + fichier)
    )

    dataframes[nom] = df

    print(f"\n--- {nom} ---")
    print("Nombre de lignes :", df.count())
    df.printSchema()
    display(df.limit(3))


In [0]:

from pyspark.sql import functions as F

df_previsions = dataframes["prevision_vs_ventes"]

# Vérifier les valeurs reçues depuis PostgreSQL
df_previsions.groupBy("semaine_a_venir").count().show()

# Convertir les valeurs texte en booléens
valeur = F.lower(F.trim(F.col("semaine_a_venir")))

df_previsions = df_previsions.withColumn(
    "semaine_a_venir",
    F.when(valeur.isin("true", "t", "1"), F.lit(True))
     .when(valeur.isin("false", "f", "0"), F.lit(False))
     .otherwise(F.lit(None).cast("boolean"))
)

# Vérifier la conversion
df_previsions.printSchema()
df_previsions.groupBy("semaine_a_venir").count().show()

print(
    "Valeurs non reconnues :",
    df_previsions.filter(F.col("semaine_a_venir").isNull()).count()
)


In [0]:

from pyspark.sql import functions as F

df_kpi_hist = dataframes["kpi_mensuel"]

print("Nombre de lignes :", df_kpi_hist.count())

df_kpi_hist.printSchema()

df_kpi_hist.agg(
    F.min("mois").alias("premier_mois"),
    F.max("mois").alias("dernier_mois"),
    F.sum("ca_mad").alias("ca_total_mad"),
    F.sum("marge_mad").alias("marge_totale_mad")
).show(truncate=False)


In [0]:

from pyspark.sql import functions as F

# Préparer les trois DataFrames
df_kpi_hist = dataframes["kpi_mensuel"]
df_stock_hist = dataframes["stock_produit"]

# df_previsions contient déjà le booléen corrigé
# Convertir les montants financiers en Decimal
df_kpi_hist = (
    df_kpi_hist
    .withColumn("ca_mad", F.col("ca_mad").cast("decimal(18,2)"))
    .withColumn("marge_mad", F.col("marge_mad").cast("decimal(18,2)"))
    .withColumn("panier_moyen_mad",
                F.col("panier_moyen_mad").cast("decimal(18,2)"))
)

tables = {
    "historique_kpi_mensuel": df_kpi_hist,
    "historique_stock_produit": df_stock_hist,
    "historique_prevision_vs_ventes": df_previsions
}

# Enregistrer chaque DataFrame en table Delta externe
for nom, df in tables.items():

    table_name = f"dbw_agricoop_poc.gold.{nom}"

    path = (
        "abfss://gold@stagricoopdata26.dfs.core.windows.net/"
        f"{nom}_delta/"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("path", path)
        .saveAsTable(table_name)
    )

    print(f"{nom} : {spark.table(table_name).count()} lignes")
